In [22]:
import re
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd

pd.set_option("display.max_rows", 200)
pd.set_option("display.max_colwidth", 200)

DATA_DIR = Path("raw_data/")          # adjust to your layout
INFILE_NODES = DATA_DIR / "new_nodes.xlsx"
INFILE_EDGES = DATA_DIR / "old_edges.xlsx"
OUT_DIR = Path("out")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [23]:
pip install openpyxl

Note: you may need to restart the kernel to use updated packages.


In [24]:
df_nodes = pd.read_excel(
    INFILE_NODES,
    sheet_name="master",
    engine="openpyxl"
)

print("Shape:", df_nodes.shape)
df_nodes.head()

Shape: (258, 15)


,node_id,name,node_type,org_type,governance_level,geographic_scale,functional_domain,roles,fema_lifeline,url,key_contact,contact_url,summary,review_flag,review_note
0,211info,211info,organization,nonprofit_community,non_governmental,Oregon,emergency_management,coordination,NaN,https://www.211info.org/,NaN,https://www.211info.org/contact/,Provides community information and referral services that support disaster preparedness and recovery.,NaN,NaN
1,AirNG,Air National Guard,organization,government,state,U.S. National,emergency_management,infrastructure_operator,safety_security,https://www.ang.af.mil/,NaN,https://www.ang.af.mil/Contact/,"Provides airlift, logistics, and emergency support capabilities for disaster response.",yes,Generic umbrella entry; consider replacing with state-specific Air National Guard entities.
2,AlertReady,Alert Ready Emergency Alert System,program,coordination_structure,multi,Canada,emergency_management,messaging_alerts_provider,communications,https://www.alertready.ca/,NaN,https://www.alertready.ca/contact/,Delivers public emergency alerts across Canada through participating broadcasters and wireless providers.,yes,Could be modeled as a national alerting system rather than a standalone program node.
3,ASCE,American Society of Civil Engineers,organization,nonprofit_community,non_governmental,U.S. National,infrastructure_systems,knowledge_provider,NaN,https://www.asce.org/,NaN,https://www.asce.org/contact,Advances engineering practice and guidance relevant to resilient infrastructure and seismic risk reduction.,NaN,NaN
4,BCHydro,BC Hydro,organization,quasi_governmental,state,British Columbia,infrastructure_systems,infrastructure_operator,energy,https://www.bchydro.com/,NaN,https://www.bchydro.com/contact.html,Operates electric generation and transmission infrastructure essential to resilience and recovery.,NaN,NaN


## node id cleaning

In [25]:
df = df_nodes.copy()

missing_node_id = df["node_id"].isna() | df["node_id"].astype(str).str.strip().eq("")
dropped_rows = df.loc[missing_node_id].copy()

print("Number of rows that will be dropped:", len(dropped_rows))
print()
print("Names being dropped:")
display(dropped_rows["name"].fillna("(missing name)").value_counts(dropna=False))

df = df.loc[~missing_node_id].copy()

print()
print("Shape after dropping missing node_id:", df.shape)

Number of rows that will be dropped: 0

Names being dropped:


Series([], Name: name, dtype: int64)


Shape after dropping missing node_id: (258, 15)


In [26]:
df["node_id"] = (
    df["node_id"]
    .astype("string")
    .str.strip()
    .str.replace(r"\s+", "", regex=True)
)

dup_node_id = df[df.duplicated("node_id", keep=False)].sort_values("node_id")

print("Number of rows with duplicated node_id:", len(dup_node_id))
display(dup_node_id)

Number of rows with duplicated node_id: 0


,node_id,name,node_type,org_type,governance_level,geographic_scale,functional_domain,roles,fema_lifeline,url,key_contact,contact_url,summary,review_flag,review_note


In [27]:
MISSING_TOKENS = {"", "nan", "none", "n/a", "na", "null", "???"}

def normalize_text(value):
    if pd.isna(value):
        return ""
    s = str(value).strip()
    s = re.sub(r"\s+", " ", s)
    return "" if s.lower() in MISSING_TOKENS else s

def pretty_label(token):
    token = normalize_text(token)
    if not token:
        return "Unknown"
    if "_" in token:
        return token.replace("_", " ").title()
    if token.islower():
        return token.title()
    return token

def split_categories(value):
    s = normalize_text(value)
    if not s:
        return ["Unknown"]

    parts = re.split(r"\s*[,;/|]\s*", s)
    out = []
    for part in parts:
        label = pretty_label(part)
        if label not in out:
            out.append(label)
    return out or ["Unknown"]

display(df[["name", "node_id"]].head(20))

,name,node_id
0,211info,211info
1,Air National Guard,AirNG
2,Alert Ready Emergency Alert System,AlertReady
3,American Society of Civil Engineers,ASCE
4,BC Hydro,BCHydro
5,Bear River Band of the Rohnerville Rancheria,BearRiver
6,Big Lagoon Rancheria,BigLagoon
7,Blue Lake Rancheria,BlueLakeRan
8,BNSF,BNSF
9,Boise State University,BoiseState


## categorical fields

In [28]:
CATEGORY_COLUMNS = [
    "node_type",
    "org_type",
    "governance_level",
    "geographic_scale",
    "functional_domain",
    "roles",
    "fema_lifeline",
]

for col in CATEGORY_COLUMNS:
    print(f"{col}: {df[col].nunique(dropna=True)} unique")
    display(df[col].fillna("(missing)").value_counts(dropna=False).head(20))
    print()

node_type: 3 unique


organization    224
hub              19
program          15
Name: node_type, dtype: int64


org_type: 8 unique


government                81
tribal_sovereign          43
coordination_structure    34
academic                  31
nonprofit_community       29
private_sector            21
quasi_governmental        13
media                      6
Name: org_type, dtype: int64


governance_level: 7 unique


...

earthquake_science        68
emergency_management      67
community_resilience      65
infrastructure_systems    58
Name: functional_domain, dtype: int64


roles: 27 unique


(missing)                                          59
coordination                                       45
knowledge_provider                                 34
infrastructure_operator                            29
policy_maker_regulator                             21
coordination, policy_maker_regulator                8
messaging_alerts_provider                           8
coordination, knowledge_provider                    8
infrastructure_operator, policy_maker_regulator     6
data_tools_provider, knowledge_provider             6
data_tools_provider                                 5
knowledge_provider, data_tools_provider             5
coordination, data_tools_provider                   4
emergency_response                                  4
coordination, messaging_alerts_provider             2
funding_provider                                    2
knowledge_provider, policy_maker_regulator          1
messaging_alerts_provider, data_tools_provider      1
infrastructure_operator, coo


fema_lifeline: 9 unique


(missing)                 190
transportation             17
communications             15
energy                     10
safety_security             9
water_systems               8
health_medical              4
energy, communications      2
energy, water_systems       2
food_hydration_shelter      1
Name: fema_lifeline, dtype: int64

In [29]:
for col in CATEGORY_COLUMNS:
    df[f"{col}_clean"] = df[col].apply(normalize_text)

df[[f"{col}_clean" for col in CATEGORY_COLUMNS]].head()

,node_type_clean,org_type_clean,governance_level_clean,geographic_scale_clean,functional_domain_clean,roles_clean,fema_lifeline_clean
0,organization,nonprofit_community,non_governmental,Oregon,emergency_management,coordination,
1,organization,government,state,U.S. National,emergency_management,infrastructure_operator,safety_security
2,program,coordination_structure,multi,Canada,emergency_management,messaging_alerts_provider,communications
3,organization,nonprofit_community,non_governmental,U.S. National,infrastructure_systems,knowledge_provider,
4,organization,quasi_governmental,state,British Columbia,infrastructure_systems,infrastructure_operator,energy


In [30]:
CATEGORY_EXPORTS = {
    "node_type": ("nodeTypes", "nodeTypePrimary"),
    "org_type": ("orgTypes", "orgTypePrimary"),
    "governance_level": ("governanceLevels", "governanceLevelPrimary"),
    "geographic_scale": ("geoTags", "geoPrimary"),
    "functional_domain": ("functionalDomains", "functionalDomainPrimary"),
    "roles": ("roleTags", "rolePrimary"),
    "fema_lifeline": ("lifelineTags", "femaLifelinePrimary"),
}

In [31]:
for source_col, (list_col, primary_col) in CATEGORY_EXPORTS.items():
    clean_col = f"{source_col}_clean"
    df[list_col] = df[clean_col].apply(split_categories)
    df[primary_col] = df[list_col].str[0]

df[[
    "name",
    "node_id",
    "orgTypePrimary",
    "geoPrimary",
    "nodeTypePrimary",
    "governanceLevelPrimary",
    "functionalDomainPrimary",
    "rolePrimary",
    "femaLifelinePrimary",
]].head(20)

,name,node_id,orgTypePrimary,geoPrimary,nodeTypePrimary,governanceLevelPrimary,functionalDomainPrimary,rolePrimary,femaLifelinePrimary
0,211info,211info,Nonprofit Community,Oregon,Organization,Non Governmental,Emergency Management,Coordination,Unknown
1,Air National Guard,AirNG,Government,U.S. National,Organization,State,Emergency Management,Infrastructure Operator,Safety Security
2,Alert Ready Emergency Alert System,AlertReady,Coordination Structure,Canada,Program,Multi,Emergency Management,Messaging Alerts Provider,Communications
3,American Society of Civil Engineers,ASCE,Nonprofit Community,U.S. National,Organization,Non Governmental,Infrastructure Systems,Knowledge Provider,Unknown
4,BC Hydro,BCHydro,Quasi Governmental,British Columbia,Organization,State,Infrastructure Systems,Infrastructure Operator,Energy
5,Bear River Band of the Rohnerville Rancheria,BearRiver,Tribal Sovereign,California,Organization,Sovereign,Community Resilience,Unknown,Unknown
6,Big Lagoon Rancheria,BigLagoon,Tribal Sovereign,California,Organization,Sovereign,Community Resilience,Unknown,Unknown
7,Blue Lake Rancheria,BlueLakeRan,Tribal Sovereign,California,Organization,Sovereign,Community Resilience,Coordination,Unknown
8,BNSF,BNSF,Private Sector,U.S. National,Organization,Private,Infrastructure Systems,Infrastructure Operator,Transportation
9,Boise State University,BoiseState,Academic,U.S. National,Organization,Non Governmental,Earthquake Science,Knowledge Provider,Unknown


In [32]:
for source_col, (list_col, primary_col) in CATEGORY_EXPORTS.items():
    labels = sorted({label for values in df[list_col] for label in values})
    print(primary_col)
    print(labels)
    print()

display(df[["orgTypePrimary", "geoPrimary"]].value_counts().head(20))

nodeTypePrimary
['Hub', 'Organization', 'Program']

orgTypePrimary
['Academic', 'Coordination Structure', 'Government', 'Media', 'Nonprofit Community', 'Private Sector', 'Quasi Governmental', 'Tribal Sovereign']

governanceLevelPrimary
['Federal', 'Local', 'Multi', 'Non Governmental', 'Private', 'Sovereign', 'State']

geoPrimary
['British Columbia', 'California', 'Canada', 'International', 'Oregon', 'PNW Regional', 'U.S. Federal', 'U.S. National', 'Washington']

functionalDomainPrimary
['Community Resilience', 'Earthquake Science', 'Emergency Management', 'Infrastructure Systems']

rolePrimary
['Coordination', 'Data Tools Provider', 'Emergency Response', 'Funding Provider', 'Infrastructure Operator', 'Knowledge Provider', 'Messaging Alerts Provider', 'Policy Maker Regulator', 'Unknown']

femaLifelinePrimary
['Communications', 'Energy', 'Food Hydration Shelter', 'Health Medical', 'Safety Security', 'Transportation', 'Unknown', 'Water Systems']



orgTypePrimary          geoPrimary      
Government              Oregon              30
Tribal Sovereign        Washington          24
Government              Washington          17
Academic                U.S. National       16
Government              U.S. Federal        16
Private Sector          U.S. National       14
Nonprofit Community     U.S. National       11
Tribal Sovereign        California          11
Coordination Structure  U.S. National       11
                        PNW Regional        10
Government              California          10
Nonprofit Community     Oregon               9
Quasi Governmental      Oregon               6
Academic                Oregon               6
Tribal Sovereign        U.S. National        6
Coordination Structure  Oregon               4
Government              British Columbia     4
Quasi Governmental      Washington           4
Nonprofit Community     International        3
Government              Canada               3
dtype: int64

## notes and contact fields

In [33]:
TEXT_EXPORT_COLUMNS = [
    "name",
    "url",
    "key_contact",
    "contact_url",
    "summary",
    "review_flag",
    "review_note",
]

for col in TEXT_EXPORT_COLUMNS:
    df[col] = df[col].apply(normalize_text)

In [34]:
display(df[[
    "name",
    "summary",
    "review_flag",
    "review_note",
    "url",
    "key_contact",
    "contact_url",
]].head(10))

,name,summary,review_flag,review_note,url,key_contact,contact_url
0,211info,Provides community information and referral services that support disaster preparedness and recovery.,,,https://www.211info.org/,,https://www.211info.org/contact/
1,Air National Guard,"Provides airlift, logistics, and emergency support capabilities for disaster response.",yes,Generic umbrella entry; consider replacing with state-specific Air National Guard entities.,https://www.ang.af.mil/,,https://www.ang.af.mil/Contact/
2,Alert Ready Emergency Alert System,Delivers public emergency alerts across Canada through participating broadcasters and wireless providers.,yes,Could be modeled as a national alerting system rather than a standalone program node.,https://www.alertready.ca/,,https://www.alertready.ca/contact/
3,American Society of Civil Engineers,Advances engineering practice and guidance relevant to resilient infrastructure and seismic risk reduction.,,,https://www.asce.org/,,https://www.asce.org/contact
4,BC Hydro,Operates electric generation and transmission infrastructure essential to resilience and recovery.,,,https://www.bchydro.com/,,https://www.bchydro.com/contact.html
5,Bear River Band of the Rohnerville Rancheria,"Provides tribal leadership, planning, and community resilience functions relevant to earthquake and hazard preparedness.",,,,,
6,Big Lagoon Rancheria,"Provides tribal leadership, planning, and community resilience functions relevant to earthquake and hazard preparedness.",,,,,
7,Blue Lake Rancheria,Leads tribal and community resilience initiatives including hazard preparedness and emergency planning.,,,,,
8,BNSF,Operates freight rail infrastructure that is critical to regional transportation resilience.,,,https://www.bnsf.com/,,https://www.bnsf.com/contact-us/
9,Boise State University,"Conducts research and education relevant to hazards, resilience, and emergency management.",,,https://www.boisestate.edu/,,https://www.boisestate.edu/contact/


In [35]:
def combine_notes(summary, review_flag, review_note):
    parts = []

    summary = normalize_text(summary)
    review_flag = normalize_text(review_flag)
    review_note = normalize_text(review_note)

    if summary:
        parts.append(summary)
    if review_flag:
        parts.append(f"Review flag: {review_flag}")
    if review_note:
        parts.append(f"Review note: {review_note}")

    return "\n\n".join(parts)

df["Organization Name"] = df["name"]
df["Org ID"] = df["node_id"]
df["Notes"] = df.apply(
    lambda row: combine_notes(row["summary"], row["review_flag"], row["review_note"]),
    axis=1,
)
df["Primary"] = df["key_contact"]
df["2ndry"] = df["contact_url"]

display(df[["Organization Name", "Org ID", "orgTypePrimary", "geoPrimary", "Notes", "Primary", "2ndry"]].head(10))

,Organization Name,Org ID,orgTypePrimary,geoPrimary,Notes,Primary,2ndry
0,211info,211info,Nonprofit Community,Oregon,Provides community information and referral services that support disaster preparedness and recovery.,,https://www.211info.org/contact/
1,Air National Guard,AirNG,Government,U.S. National,"Provides airlift, logistics, and emergency support capabilities for disaster response.\n\nReview flag: yes\n\nReview note: Generic umbrella entry; consider replacing with state-specific Air Nation...",,https://www.ang.af.mil/Contact/
2,Alert Ready Emergency Alert System,AlertReady,Coordination Structure,Canada,Delivers public emergency alerts across Canada through participating broadcasters and wireless providers.\n\nReview flag: yes\n\nReview note: Could be modeled as a national alerting system rather ...,,https://www.alertready.ca/contact/
3,American Society of Civil Engineers,ASCE,Nonprofit Community,U.S. National,Advances engineering practice and guidance relevant to resilient infrastructure and seismic risk reduction.,,https://www.asce.org/contact
4,BC Hydro,BCHydro,Quasi Governmental,British Columbia,Operates electric generation and transmission infrastructure essential to resilience and recovery.,,https://www.bchydro.com/contact.html
5,Bear River Band of the Rohnerville Rancheria,BearRiver,Tribal Sovereign,California,"Provides tribal leadership, planning, and community resilience functions relevant to earthquake and hazard preparedness.",,
6,Big Lagoon Rancheria,BigLagoon,Tribal Sovereign,California,"Provides tribal leadership, planning, and community resilience functions relevant to earthquake and hazard preparedness.",,
7,Blue Lake Rancheria,BlueLakeRan,Tribal Sovereign,California,Leads tribal and community resilience initiatives including hazard preparedness and emergency planning.,,
8,BNSF,BNSF,Private Sector,U.S. National,Operates freight rail infrastructure that is critical to regional transportation resilience.,,https://www.bnsf.com/contact-us/
9,Boise State University,BoiseState,Academic,U.S. National,"Conducts research and education relevant to hazards, resilience, and emergency management.",,https://www.boisestate.edu/contact/


In [36]:
df[[
    "Organization Name",
    "Org ID",
    "orgTypes",
    "orgTypePrimary",
    "geoTags",
    "geoPrimary",
    "nodeTypePrimary",
    "governanceLevelPrimary",
    "functionalDomainPrimary",
    "rolePrimary",
    "femaLifelinePrimary",
]].head(3)

,Organization Name,Org ID,orgTypes,orgTypePrimary,geoTags,geoPrimary,nodeTypePrimary,governanceLevelPrimary,functionalDomainPrimary,rolePrimary,femaLifelinePrimary
0,211info,211info,[Nonprofit Community],Nonprofit Community,[Oregon],Oregon,Organization,Non Governmental,Emergency Management,Coordination,Unknown
1,Air National Guard,AirNG,[Government],Government,[U.S. National],U.S. National,Organization,State,Emergency Management,Infrastructure Operator,Safety Security
2,Alert Ready Emergency Alert System,AlertReady,[Coordination Structure],Coordination Structure,[Canada],Canada,Program,Multi,Emergency Management,Messaging Alerts Provider,Communications


In [37]:
import json

df["Org ID"] = df["Org ID"].astype(str)

for col in [
    "orgTypes",
    "geoTags",
    "nodeTypes",
    "governanceLevels",
    "functionalDomains",
    "roleTags",
    "lifelineTags",
]:
    df[f"{col}_json"] = df[col].apply(json.dumps)

cols = [
    "Organization Name",
    "Org ID",
    "orgTypes_json",
    "orgTypePrimary",
    "geoPrimary",
    "Notes",
    "Primary",
    "2ndry",
    "geoTags_json",
    "nodeTypes_json",
    "nodeTypePrimary",
    "governanceLevels_json",
    "governanceLevelPrimary",
    "functionalDomains_json",
    "functionalDomainPrimary",
    "roleTags_json",
    "rolePrimary",
    "lifelineTags_json",
    "femaLifelinePrimary",
    "url",
    "review_flag",
    "review_note",
]

clean_df = df[cols].copy()

clean_df.to_csv("organizations_clean.csv", index=False)

print("Exported:", len(clean_df), "rows")
clean_df.head()

Exported: 258 rows


,Organization Name,Org ID,orgTypes_json,orgTypePrimary,geoPrimary,Notes,Primary,2ndry,geoTags_json,nodeTypes_json,...,governanceLevelPrimary,functionalDomains_json,functionalDomainPrimary,roleTags_json,rolePrimary,lifelineTags_json,femaLifelinePrimary,url,review_flag,review_note
0,211info,211info,"[""Nonprofit Community""]",Nonprofit Community,Oregon,Provides community information and referral services that support disaster preparedness and recovery.,,https://www.211info.org/contact/,"[""Oregon""]","[""Organization""]",...,Non Governmental,"[""Emergency Management""]",Emergency Management,"[""Coordination""]",Coordination,"[""Unknown""]",Unknown,https://www.211info.org/,,
1,Air National Guard,AirNG,"[""Government""]",Government,U.S. National,"Provides airlift, logistics, and emergency support capabilities for disaster response.\n\nReview flag: yes\n\nReview note: Generic umbrella entry; consider replacing with state-specific Air Nation...",,https://www.ang.af.mil/Contact/,"[""U.S. National""]","[""Organization""]",...,State,"[""Emergency Management""]",Emergency Management,"[""Infrastructure Operator""]",Infrastructure Operator,"[""Safety Security""]",Safety Security,https://www.ang.af.mil/,yes,Generic umbrella entry; consider replacing with state-specific Air National Guard entities.
2,Alert Ready Emergency Alert System,AlertReady,"[""Coordination Structure""]",Coordination Structure,Canada,Delivers public emergency alerts across Canada through participating broadcasters and wireless providers.\n\nReview flag: yes\n\nReview note: Could be modeled as a national alerting system rather ...,,https://www.alertready.ca/contact/,"[""Canada""]","[""Program""]",...,Multi,"[""Emergency Management""]",Emergency Management,"[""Messaging Alerts Provider""]",Messaging Alerts Provider,"[""Communications""]",Communications,https://www.alertready.ca/,yes,Could be modeled as a national alerting system rather than a standalone program node.
3,American Society of Civil Engineers,ASCE,"[""Nonprofit Community""]",Nonprofit Community,U.S. National,Advances engineering practice and guidance relevant to resilient infrastructure and seismic risk reduction.,,https://www.asce.org/contact,"[""U.S. National""]","[""Organization""]",...,Non Governmental,"[""Infrastructure Systems""]",Infrastructure Systems,"[""Knowledge Provider""]",Knowledge Provider,"[""Unknown""]",Unknown,https://www.asce.org/,,
4,BC Hydro,BCHydro,"[""Quasi Governmental""]",Quasi Governmental,British Columbia,Operates electric generation and transmission infrastructure essential to resilience and recovery.,,https://www.bchydro.com/contact.html,"[""British Columbia""]","[""Organization""]",...,State,"[""Infrastructure Systems""]",Infrastructure Systems,"[""Infrastructure Operator""]",Infrastructure Operator,"[""Energy""]",Energy,https://www.bchydro.com/,,


# looking at edges

In [38]:
df_edges = pd.read_excel(
    INFILE_EDGES,
    sheet_name="Relationships",
    engine="openpyxl"
)

print("Shape:", df_edges.shape)
df_edges.head()

Shape: (536, 5)


,From agency,To agency,Relationship type,Description,Status
0,BCHydro,NRCanGSC,data,NaN,NaN
1,BCHydro,NRCanGSC,tools/products,sharing models,NaN
2,NRCanGSC,BCHydro,data,NaN,NaN
3,NRCanGSC,BCHydro,tools/products,sharing models,NaN
4,CRESCENT,BCHydro,data,NaN,NaN


In [39]:
df_edges = df_edges.copy()

def clean_id(x):
    if pd.isna(x):
        return pd.NA
    # remove all whitespace, then (optional) strip punctuation the same way as nodes
    s = str(x).strip()
    s = pd.Series([s]).str.replace(r"\s+", "", regex=True).iloc[0]
    return s

df_edges["From agency"] = df_edges["From agency"].apply(clean_id)
df_edges["To agency"]   = df_edges["To agency"].apply(clean_id)

# Optional: also normalize Relationship type / Description / Status whitespace
for c in ["Relationship type", "Description", "Status"]:
    if c in df_edges.columns:
        df_edges[c] = (df_edges[c].astype("string")
                                   .str.strip()
                                   .str.replace(r"\s+", " ", regex=True))

# Drop *exact* duplicate rows (keeps first occurrence)
before = len(df_edges)
df_edges_clean = df_edges.drop_duplicates(keep="first").reset_index(drop=True)
after = len(df_edges_clean)

print(f"Rows before: {before}  |  after drop_duplicates: {after}  |  removed: {before-after}")
df_edges_clean.head()

Rows before: 536  |  after drop_duplicates: 520  |  removed: 16


,From agency,To agency,Relationship type,Description,Status
0,BCHydro,NRCanGSC,data,<NA>,<NA>
1,BCHydro,NRCanGSC,tools/products,sharing models,<NA>
2,NRCanGSC,BCHydro,data,<NA>,<NA>
3,NRCanGSC,BCHydro,tools/products,sharing models,<NA>
4,CRESCENT,BCHydro,data,<NA>,<NA>


In [40]:
df_edges_clean.to_csv("edges_clean.csv", index=False)
print("Wrote edges_clean.csv")

Wrote edges_clean.csv


# verify alignment

In [41]:
node_ids = set(clean_df["Org ID"].astype(str))

missing_from = sorted(set(df_edges_clean["From agency"]) - node_ids)
missing_to   = sorted(set(df_edges_clean["To agency"]) - node_ids)

print("edges FROM not in nodes:", missing_from)
print("----------------------")
print("edges TO not in nodes:", missing_to)

edges FROM not in nodes: ['ASF', 'BCAlert', 'BCEM', 'CAmedia', 'CaSSC', 'CalDWR', 'CalEAS', 'CalFire', 'CalTrans', 'ClackCo', 'Copes', 'DNR', 'EERi', 'IEMA', 'JPL', 'LidarBC', 'NHRP', 'NOAA', 'NOAA-NTWC', 'NOAA-NWS', 'NOAANWS', 'NOAANWTC', 'NRCanGSC', 'NTWC', 'NWS', 'ODHS', 'OR-Gov', 'ORDEQ', 'OREAS', 'OREM', 'ORMedia', 'ORmedia', 'PDXOEM', 'PMEL', 'PTWC', 'PWB', 'PacificCor', 'RCTWG', 'SeaGrant', 'Tribal', 'Trimet', 'UP', 'USGA', 'USGS', 'WADNRWGS', 'WAEAS', 'WAmedia', 'WPUC', 'copes', 'dogami', 'eeri', 'oem']
----------------------
edges TO not in nodes: ['BCEM', 'CAmedia', 'CaSSC', 'CalDWR', 'CalFire', 'CalTrans', 'ClackCo', 'Copes', 'DLCD', 'DNR', 'Earthscope', 'FIN', 'JPL', 'NHRP', 'NOAA', 'NOAA-NTWC', 'NOAA-NWS', 'NRCanGSC', 'NWS', 'ODHS', 'OR-Gov', 'ORDEQ', 'OREAS', 'OREM', 'ORMedia', 'ORmedia', 'Osspac', 'PDXOEM', 'PSC', 'PWB', 'RCTWG', 'SeaGrant', 'Sz4d', 'Tribal', 'Trimet', 'UP', 'USGS', 'WADNRWGS', 'WAmedia', 'WSDOT', 'copes', 'dogami', 'oem']


In [42]:
node_ids = set(clean_df["Org ID"].astype(str))

missing_from = sorted(node_ids - set(df_edges_clean["From agency"]))
missing_to   = sorted(node_ids - set(df_edges_clean["To agency"]))

print("Nodes not in edges From:", missing_from)
print("----------------------")

print("Nodes not in edges To", missing_to)

Nodes not in edges From: ['211info', 'ASCE', 'AirNG', 'AlertFM', 'AlertReady', 'BCEMCR', 'BCTransport', 'BLM', 'Bainbridge', 'BearRiver', 'BellPort', 'BigLagoon', 'BlueLakeRan', 'BoiseState', 'CACC', 'CADWR', 'CALFIRE', 'CAParks', 'CBC', 'CERT', 'CHRN', 'CICOES', 'CLPUD', 'CLaSH', 'COHORT', 'COStateU', 'CPBR', 'CPHumboldt', 'CRESA', 'CSM', 'CSSC', 'CTCLUSI', 'CalAlerts', 'CalGuard', 'Caltrans', 'CascadiaCC', 'Chehalis', 'ClackamasEM', 'ConsejoHisp', 'CoosEM', 'Coquille', 'CowCreek', 'Cowlitz', 'EMCowichan', 'EPA', 'EPS', 'ESC', 'ESCERT', 'EVACNR', 'EWLabs', 'EarthScope', 'ElkValley', 'Eugene', 'Facet', 'FedMedia', 'FinCan', 'FraserBC', 'GHI', 'GLAD', 'GSC', 'GSOC', 'GovOR', 'GrandRonde', 'HaleyAldr', 'Harvard', 'Hoh', 'HomeForward', 'Hoopa', 'IAEE', 'IDStateU', 'IPREM', 'ImageCat', 'IndianaU', 'Jamestown', 'KMP', 'Karuk', 'KingCoEM', 'KyotoU', 'L&Ccollege', 'LaneCoEM', 'LincolnCity', 'LinkOregon', 'LinntonNA', 'LowerElwha', 'Lummi', 'MHCC', 'Makah', 'Moonshot', 'Muckleshoot', 'Multnoma

In [44]:
edge_endpoints = set(df_edges_clean["From agency"].dropna()) | set(df_edges_clean["To agency"].dropna())
org_ids = sorted(clean_df["Org ID"].dropna().astype(str).unique())

org_ids_with_edges = sorted([org_id for org_id in org_ids if org_id in edge_endpoints])
org_ids_without_edges = sorted([org_id for org_id in org_ids if org_id not in edge_endpoints])

summary_df = pd.DataFrame(
    {
        "status": ["has at least one edge", "has no edges"],
        "count": [len(org_ids_with_edges), len(org_ids_without_edges)],
        "share": [
            len(org_ids_with_edges) / len(org_ids) if org_ids else 0,
            len(org_ids_without_edges) / len(org_ids) if org_ids else 0,
        ],
    }
)

display(summary_df)

print("Org IDs with edges:")
print(org_ids_with_edges)
print()

print("Org IDs without edges")
print(org_ids_without_edges)
print()

display(
    clean_df.assign(has_edges=clean_df["Org ID"].astype(str).isin(edge_endpoints))
    .sort_values(["has_edges", "Org ID"], ascending=[False, True])
    [["Org ID", "Organization Name", "orgTypePrimary", "geoPrimary", "has_edges"]]
    .reset_index(drop=True)
)

,status,count,share
0,has at least one edge,51,0.197674
1,has no edges,207,0.802326


Org IDs with edges:
['BCHydro', 'BNSF', 'BPA', 'CGS', 'CLiP', 'CPUC', 'CRESCENT', 'CREW', 'CalOES', 'CoPesHub', 'DOGAMI', 'ECA', 'EEI', 'EERI', 'EWEB', 'FEMA', 'GEM', 'GLAD', 'GeoBC', 'Hakai', 'IAEE', 'MetroVan', 'NEHRP', 'NEMA', 'NHERI', 'NIFC', 'NTHMP', 'ODFW', 'ODOT', 'OEM', 'OHA', 'OMSI', 'OPRD', 'OPUC', 'OSSPAC', 'PACICC', 'PBEM', 'PBOT', 'PEER', 'PG&E', 'PGE', 'PNSN', 'PacifiCorp', 'RDPO', 'SCEC', 'SalemOEM', 'USACE', 'USBR', 'USFS', 'WAEMD', 'WEI']

Org IDs without edges
['211info', 'ASCE', 'AirNG', 'AlertFM', 'AlertReady', 'BCEMCR', 'BCTransport', 'BLM', 'Bainbridge', 'BearRiver', 'BellPort', 'BigLagoon', 'BlueLakeRan', 'BoiseState', 'CACC', 'CADWR', 'CALFIRE', 'CAParks', 'CBC', 'CERT', 'CHRN', 'CICOES', 'CLPUD', 'CLaSH', 'COHORT', 'COStateU', 'CPBR', 'CPHumboldt', 'CRESA', 'CSM', 'CSSC', 'CTCLUSI', 'CalAlerts', 'CalGuard', 'Caltrans', 'CascadiaCC', 'Chehalis', 'ClackamasEM', 'ConsejoHisp', 'CoosEM', 'Coquille', 'CowCreek', 'Cowlitz', 'EMCowichan', 'EPA', 'EPS', 'ESC', 'ESCERT'

,Org ID,Organization Name,orgTypePrimary,geoPrimary,has_edges
0,BCHydro,BC Hydro,Quasi Governmental,British Columbia,True
1,BNSF,BNSF,Private Sector,U.S. National,True
2,BPA,Bonneville Power Administration,Quasi Governmental,PNW Regional,True
3,CGS,California Geological Survey,Government,California,True
4,CLiP,Cascadia Lifelines Program,Coordination Structure,PNW Regional,True
...,...,...,...,...,...
253,WWU,Western Washington University,Academic,Washington,False
254,WashPost,Washington Post,Media,U.S. National,False
255,Wiyot,Wiyot Tribe,Tribal Sovereign,California,False
256,Yakama,Confederated Tribes and Bands of the Yakama Nation,Tribal Sovereign,U.S. National,False
